# NIPOPETO — kiểm thử tải Pha 1 trên Kaggle CPU

Upload notebook này lên Kaggle, chọn **Accelerator: None**, bật **Internet**, chạy các cell theo thứ tự. Backend, PostgreSQL và Locust dùng chung CPU; không cần Docker hoặc database của máy cá nhân.

**Trước khi chạy:** commit/push cả mã ứng dụng và các file harness vừa sửa, rồi lấy `git rev-parse HEAD` và điền full SHA 40 ký tự. Notebook kiểm tra protocol của source để tránh chạy runner cũ bằng cấu hình mới. Không đưa `.env`, dữ liệu thật hay khóa bí mật vào notebook.

Mỗi run dùng database tạm riêng và ít nhất 100 cặp chủ nuôi/thú cưng giả. Sau khi xong hoặc có run lỗi, vẫn chạy cell cuối để tạo ZIP và tải từ mục **Output** (refresh nếu chưa thấy). Đây là đo backend; chưa bao phủ mọi hành trình admin/thanh toán hoặc giao diện frontend.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, platform, pwd, re, shutil, signal, subprocess, sys, urllib.request, uuid, zipfile

REPO_URL = "https://github.com/Nguyenquanghong/Pet-healthcare.git"
COMMIT_SHA = "DIEN_FULL_SHA_40_KY_TU_SAU_KHI_PUSH"
RUN_LABEL = "B1"  # B2 cho bản cải tiến, giữ cùng protocol và cấu hình đo.
PROFILES = ("ReadHeavyUser", "MixedUser")
USER_LEVELS = (1, 10, 50, 100)
REPEATS = 2
SPAWN_RATE = 5
WARMUP_SECONDS = 45
DURATION_SECONDS = 120  # Tổng thời gian process đo, gồm ramp-up.
PROTOCOL = "phase1-kaggle-v2"
WORKING = Path("/kaggle/working")

assert WORKING.is_dir(), "Notebook này phải chạy trên Kaggle."
assert not os.environ.get("COLAB_GPU") and not Path("/dev/nvidia0").exists(), "Chọn Accelerator: None trước khi kiểm thử CPU."
assert re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT_SHA), "Điền full COMMIT_SHA 40 ký tự của bản đã push."
assert re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+\.git", REPO_URL), "Dùng URL GitHub public; không nhúng access token."
assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_LABEL), "RUN_LABEL chỉ gồm chữ, số, _ hoặc -."
assert REPEATS >= 1 and SPAWN_RATE > 0 and USER_LEVELS and all(isinstance(u, int) and u >= 1 for u in USER_LEVELS)
assert len(set(USER_LEVELS)) == len(USER_LEVELS) and len(set(PROFILES)) == len(PROFILES)
assert PROFILES and set(PROFILES) <= {"ReadHeavyUser", "MixedUser"}
ramp_seconds = math.ceil(max(USER_LEVELS) / SPAWN_RATE)
assert WARMUP_SECONDS >= ramp_seconds + 15, "Warmup cần đủ ramp-up và ít nhất 15s sau ramp."
assert DURATION_SECONDS >= ramp_seconds + 60, "Lượt đo cần ít nhất 60s sau ramp."
EXPECTED_RUNS = len(PROFILES) * len(USER_LEVELS) * REPEATS
repo = WORKING / "Pet-healthcare"
print("Protocol:", PROTOCOL, "| Run label:", RUN_LABEL, "| CPU logic:", os.cpu_count(), "| Source:", COMMIT_SHA)
print("Ma trận:", EXPECTED_RUNS, "run; warmup:", WARMUP_SECONDS, "s; process đo:", DURATION_SECONDS, "s (gồm ramp-up).")


## Lấy đúng source

Dùng lại checkout sạch nếu đã clone; có thể chuyển sang SHA mới khi checkout sạch. Dừng khi có thay đổi để tránh mất mã hoặc gắn sai nhãn kết quả. Không tự xóa checkout cũ. Trạng thái detached HEAD khi chọn đúng SHA là bình thường.


In [ ]:
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip()
assert not dirty, "Checkout có thay đổi; hãy kiểm tra trước khi tiếp tục."
if actual.lower() != COMMIT_SHA.lower():
    subprocess.run(["git", "fetch", "origin", COMMIT_SHA], cwd=repo, check=True)
    subprocess.run(["git", "checkout", "--detach", COMMIT_SHA], cwd=repo, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip()
assert actual.lower() == COMMIT_SHA.lower() and not dirty, "SHA không khớp hoặc working tree không sạch."
source_protocol = subprocess.check_output([sys.executable, "-c", "import sys; sys.path.insert(0, sys.argv[1]); from metrics import BENCHMARK_PROTOCOL; print(BENCHMARK_PROTOCOL)", str(repo / "benchmarks/kaggle")], text=True).strip()
assert source_protocol == PROTOCOL, "Source chưa chứa harness hiện tại. Commit/push notebook + runner + locustfile + metrics rồi chọn lại SHA."
print("Source commit:", actual, "| dirty_worktree:", bool(dirty), "| protocol:", source_protocol)


## Cài công cụ và kiểm tra trước chạy

Cần Internet để tải PostgreSQL server, Node.js 24 và Python packages. Cài Python bằng chính interpreter của kernel. Cell kiểm đủ `initdb`, `pg_ctl`, `psql`, `createdb`, và tài khoản hệ thống postgres khi chạy bằng root; client PostgreSQL đơn lẻ chưa đủ.


In [ ]:
def find_pg_bin():
    candidates = []
    if shutil.which("pg_config"):
        candidates.append(Path(subprocess.check_output(["pg_config", "--bindir"], text=True).strip()))
    candidates.extend(sorted(Path("/usr/lib/postgresql").glob("*/bin"), reverse=True))
    return next((p for p in candidates if all((p / tool).is_file() for tool in ("initdb", "pg_ctl", "postgres", "psql", "createdb"))), None)

pg_bin = find_pg_bin()
if pg_bin is None or not shutil.which("pg_config"):
    assert os.geteuid() == 0, "Không có quyền cài PostgreSQL server trong phiên này."
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "postgresql", "postgresql-client", "libpq-dev"], check=True)
    pg_bin = find_pg_bin()
assert pg_bin is not None, "Chưa tìm được PostgreSQL server; xem output cài đặt."
os.environ["BENCH_PG_BIN"] = str(pg_bin)
os.environ["PATH"] = str(pg_bin) + os.pathsep + os.environ["PATH"]
if os.geteuid() == 0:
    assert shutil.which("runuser"), "Thiếu runuser để chạy PostgreSQL bằng user postgres."
    pwd.getpwnam("postgres")

assert platform.machine() in ("x86_64", "amd64"), "Bản Node pin này dành cho Kaggle Linux x64."
node_dir = WORKING / "node24"
if (node_dir / "bin/node").is_file():
    os.environ["PATH"] = str(node_dir / "bin") + os.pathsep + os.environ["PATH"]
node_ok = shutil.which("node") and subprocess.check_output(["node", "--version"], text=True).strip().startswith("v24.")
if not node_ok:
    version = "v24.15.0"
    name = f"node-{version}-linux-x64.tar.xz"
    base = f"https://nodejs.org/dist/{version}/"
    archive = WORKING / name
    urllib.request.urlretrieve(base + name, archive)
    checksums = urllib.request.urlopen(base + "SHASUMS256.txt", timeout=60).read().decode()
    expected = next(line.split()[0] for line in checksums.splitlines() if line.split()[-1] == name)
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, "Node checksum không khớp."
    node_dir.mkdir(exist_ok=True)
    subprocess.run(["tar", "-xJf", str(archive), "-C", str(node_dir), "--strip-components=1"], check=True)
    os.environ["PATH"] = str(node_dir / "bin") + os.pathsep + os.environ["PATH"]
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo / "benchmarks/kaggle/requirements.txt")], check=True)
for program, args in {"node": ["--version"], "npm": ["--version"], "pg_config": ["--version"]}.items():
    print(program, subprocess.check_output([program, *args], text=True).strip())
print("PostgreSQL server:", subprocess.check_output([str(pg_bin / "postgres"), "--version"], text=True).strip())
print("Python:", sys.version.split()[0], "| Locust:", subprocess.check_output([sys.executable, "-m", "locust", "--version"], text=True).strip())
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(repo / "benchmarks/kaggle"), "-p", "test_*.py", "-v"], check=True)


## Chạy ma trận

Mặc định hai profile × 1/10/50/100 users × hai lần = 16 run. Warmup 45s, process đo 120s, tăng 5 users/s; think time 0,5–1,5s. **Lượt đo khởi động Locust mới và vẫn có ramp-up**, nên số tổng hợp là trung bình cả cửa sổ 120s, không phải riêng steady state. 100 users có khoảng 100s sau ramp. Dữ liệu POST của warmup và measured dùng slot khác nhau.

Dừng khi run lỗi hoặc vượt guardrail error rate >1% / p95 >2.000ms. Tạo thư mục mới mỗi lần, giữ bằng chứng cũ. Thời gian tải riêng khoảng 44 phút cho 16 run, cộng thời gian cài/build/migrate. Nếu một run lỗi hoặc bạn dừng cell, vẫn chạy cell cuối để xuất kết quả một phần; không báo PASS cho ma trận chưa hoàn thành.


In [ ]:
current_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
current_dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip()
assert current_sha.lower() == COMMIT_SHA.lower() and not current_dirty, "Source thay đổi sau clone; kiểm tra trước khi đo."
evidence = WORKING / f"nipopeto-evidence-{RUN_LABEL}-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid.uuid4().hex[:8]}"
evidence.mkdir(exist_ok=False)
run_error = None
run_count = 0
protocol_config = {"profiles": PROFILES, "user_levels": USER_LEVELS, "repeats": REPEATS, "spawn_rate": SPAWN_RATE, "warmup_seconds": WARMUP_SECONDS, "duration_seconds": DURATION_SECONDS}
try:
    for profile in PROFILES:
        if run_error:
            break
        for users in USER_LEVELS:
            if run_error:
                break
            for repeat in range(1, REPEATS + 1):
                folder = evidence / f"{RUN_LABEL}_{profile}_{users}_{repeat}"
                args = [sys.executable, str(repo / "benchmarks/kaggle/runner.py"), "--repo", str(repo), "--output", str(folder), "--profile", profile, "--users", str(users), "--spawn-rate", str(SPAWN_RATE), "--warmup", f"{WARMUP_SECONDS}s", "--duration", f"{DURATION_SECONDS}s", "--seed", "phase1"]
                launcher_log = evidence / f"{folder.name}-runner.log"
                print("Running", folder.name, "| log:", launcher_log.name, flush=True)
                run_count += 1
                with launcher_log.open("w", encoding="utf-8") as log:
                    process = subprocess.Popen(args, cwd=repo, env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
                    try:
                        returncode = process.wait()
                    except KeyboardInterrupt:
                        if process.poll() is None:
                            try:
                                os.killpg(process.pid, signal.SIGINT)
                            except ProcessLookupError:
                                pass
                            try:
                                process.wait(timeout=45)
                            except subprocess.TimeoutExpired:
                                raise RuntimeError("Runner chưa dọn xong sau 45s. Đợi và kiểm tra log/port; khởi động phiên Kaggle mới nếu cần, trước khi chạy lại.")
                        raise
                print("\n".join(launcher_log.read_text(encoding="utf-8", errors="replace").splitlines()[-8:]), flush=True)
                if returncode:
                    run_error = f"{folder.name}: exit {returncode}. Xem runner log, manifest và log trong ZIP."
                    break
except KeyboardInterrupt:
    run_error = "Người dùng đã dừng cell; đây là kết quả một phần. Kiểm tra cleanup trong runner log trước khi chạy lại."
except Exception as error:
    run_error = f"Notebook orchestration lỗi: {type(error).__name__}: {error}"
finally:
    (evidence / "notebook-summary.json").write_text(json.dumps({
        "commit": actual, "label": RUN_LABEL, "benchmark_protocol": PROTOCOL, "config": protocol_config,
        "attempted_runs": run_count, "expected_runs": EXPECTED_RUNS, "stop_reason": run_error,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã thử:", run_count, "/", EXPECTED_RUNS, "| Bằng chứng:", evidence)
if run_error:
    print("Đã dừng:", run_error)


## Tổng hợp và tải kết quả

`manifest.results` lấy **bộ đếm cuối lúc Locust kết thúc** từ `locust_final.json`, tách warmup. CSV vẫn giữ để xem từng endpoint và lịch sử; `csv_snapshot` giúp thấy chênh lệch với snapshot định kỳ. p95/p99 dùng histogram làm tròn của Locust.

Cell tạo ZIP cả khi chưa đủ ma trận, kèm lý do dừng và runner log cho các lỗi sớm. Chỉ báo `COMPLETE_PASS` khi đủ mọi run và các manifest khớp SHA/protocol, sạch, exit 0, có request và không vượt ngưỡng. Dừng/thiếu/lỗi là `PARTIAL_OR_FAILED`; vẫn tải ZIP để phân tích. ZIP chỉ chứa thống kê/log, không source, `.env`, token fixture hoặc database tạm. Chạy lại cell xuất ZIP tạo tên khác, giữ file đã có.


In [ ]:
assert "evidence" in globals() and evidence.is_dir(), "Chạy cell ma trận trước để có thư mục bằng chứng."
rows = []
allowed = {"manifest.json", "locust_final.json", "load_stats.csv", "load_failures.csv", "load_stats_history.csv", "load_exceptions.csv", "resources.json", "setup.log", "api.log", "warmup.log", "load.log", "postgres.log"}
for folder in sorted(evidence.iterdir()):
    if not folder.is_dir():
        continue
    manifest = folder / "manifest.json"
    try:
        data = json.loads(manifest.read_text(encoding="utf-8")) if manifest.exists() else {}
    except (OSError, ValueError):
        data = {"status": "INVALID_MANIFEST"}
    results = data.get("results") or {}
    passed = (data.get("status") == "completed" and data.get("threshold_exceeded") is False and data.get("locust_exit_code") == 0
              and data.get("commit", "").lower() == actual.lower() and data.get("dirty_worktree") is False
              and data.get("benchmark_protocol") == PROTOCOL and results.get("stats_source") == "locust_quitting" and results.get("requests", 0) > 0)
    rows.append({"run": folder.name, "status": data.get("status", "NO_MANIFEST"), "passed": passed,
                 "requests": results.get("requests"), "failures": results.get("failures"), "rps": results.get("achieved_rps"),
                 "p95_ms": results.get("p95_ms"), "threshold_exceeded": data.get("threshold_exceeded")})
for row in rows:
    print(row)
summary_path = evidence / "notebook-summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
all_passed = (len(rows) == EXPECTED_RUNS and summary.get("attempted_runs") == EXPECTED_RUNS and not summary.get("stop_reason") and all(row["passed"] for row in rows))
summary.update({"result": "COMPLETE_PASS" if all_passed else "PARTIAL_OR_FAILED", "passed_runs": sum(row["passed"] for row in rows), "runs": rows})
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("Kết quả:", summary["result"], "| PASS:", summary["passed_runs"], "/", EXPECTED_RUNS, "| lý do dừng:", summary.get("stop_reason") or "không")

def write_artifact(output, item, member):
    if item.suffix == ".log":
        safe = item.read_text(encoding="utf-8", errors="replace")
        safe = re.sub(r"postgres(?:ql)?://[^\s'\"]+", "[REDACTED_DATABASE_URL]", safe, flags=re.IGNORECASE)
        safe = re.sub(r"Bearer\s+[A-Za-z0-9._~-]+", "Bearer [REDACTED]", safe, flags=re.IGNORECASE)
        output.writestr(member, safe)
    else:
        output.write(item, member)

archive = evidence.with_suffix(".zip")
if archive.exists():
    archive = evidence.with_name(f"{evidence.name}-export-{uuid.uuid4().hex[:8]}.zip")
with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as output:
    write_artifact(output, summary_path, f"{evidence.name}/{summary_path.name}")
    for item in sorted(evidence.iterdir()):
        if item.is_file() and item.name.endswith("-runner.log"):
            write_artifact(output, item, f"{evidence.name}/{item.name}")
        elif item.is_dir():
            for artifact in sorted(item.iterdir()):
                if artifact.is_file() and artifact.name in allowed:
                    write_artifact(output, artifact, f"{evidence.name}/{item.name}/{artifact.name}")
assert archive.exists() and archive.stat().st_size > 0
print("Tải ZIP từ mục Output:", archive, "| bytes:", archive.stat().st_size)
